# Load dependencies

first run this command `uv add datasets` in the terminal

In [ ]:
from datasets import load_dataset
from pprint import pprint
import html

# Loading a local dataset

In [ ]:
# !wget -P data https://github.com/crux82/squad-it/raw/master/SQuAD_it-train.json.gz
# !wget -P data https://github.com/crux82/squad-it/raw/master/SQuAD_it-test.json.gz
# !gzip -dkv data/SQuAD_it-*.json.gz

- To load a JSON file with the `load_dataset()` function, we just need to know if we’re dealing with ordinary **JSON** (similar to a nested dictionary) or **JSON Lines** (line-separated JSON).
- SQuAD-it uses the nested format, with all the text stored in a `data` field.
- This means we can load the dataset by specifying the `field` argument as follows:

## A single file path --> defaults to `train` split

In [ ]:
squad_it_dataset = load_dataset("json", data_files="data/SQuAD_it-train.json", field="data")
print(squad_it_dataset)

sample = squad_it_dataset["train"][0]
print(sample.keys())
print(sample['title'])
print(len(sample['paragraphs']))
pprint(sample['paragraphs'][0])

In [ ]:
# this would also put all data parts in the train split.
squad_it_dataset = load_dataset("json", data_files="data/*.json", field="data")
print(squad_it_dataset)

## A dict of file paths

In [ ]:
data_files = {"train": "data/SQuAD_it-train.json", "test": "data/SQuAD_it-test.json"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
print(squad_it_dataset)

the `load_dataset` also handles the unzipping automatically

In [ ]:
data_files = {"train": "data/SQuAD_it-train.json.gz", "test": "data/SQuAD_it-test.json.gz"}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
print(squad_it_dataset)

# Loading a remote dataset

In [ ]:
url = "https://github.com/crux82/squad-it/raw/master/"
data_files = {
    "train": url + "SQuAD_it-train.json.gz",
    "test": url + "SQuAD_it-test.json.gz",
}
squad_it_dataset = load_dataset("json", data_files=data_files, field="data")
print(squad_it_dataset)

# Data preprocessing (slice and dice)

In [ ]:
# data_path = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
# !wget -P data {data_path}
# !unzip -o data/drugsCom_raw.zip -d data/

In [ ]:
data_files = {"train": "data/drugsComTrain_raw.tsv", "test": "data/drugsComTest_raw.tsv"}
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
print(drug_dataset)

## Sampling for initial review

In [ ]:
drug_sample = drug_dataset["train"]\
    .shuffle(seed=42)\
    .select(range(1000)) # expects an iterable of indices
drug_sample[:3]

- The `Unnamed: 0` column looks suspiciously like an anonymized ID for each patient.
- The `condition` column includes a mix of uppercase and lowercase labels.
- The `reviews` are of varying length and contain a mix of Python line separators (`\r\n`) as well as HTML character codes like `&\#039;`.

## Addressing each issue with a proper preprocessing solution

1. Verify that the number of IDs matches the number of rows in each split.
- if it's verified, rename the ID column to a proper name

In [ ]:
print(drug_dataset.keys())
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

# # find the number of unique drugs and conditions in the training and test sets.
# for split in drug_dataset.keys():
#     print(f"split: {split}")
#     for col in ["condition", "drugName"]:
#         unique_count = drug_dataset[split].unique(col).__sizeof__()
#         print(f"number of {col} unique values: {unique_count}")

In [ ]:
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
print(drug_dataset)

2. Next, let’s normalize all the `condition` labels using `Dataset.map()`.

In [ ]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}
# drug_dataset.map(lowercase_condition)
# AttributeError: 'NoneType' object has no attribute 'lower'
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)
drug_dataset = drug_dataset.map(lowercase_condition)

In [ ]:
(drug_dataset["train"]["condition"][:3])

3. Handling `review` length distribution
- Whenever you’re dealing with customer reviews, a good practice is to check the number of words in each review.
- A review might be just a single word like “Great!” or a full-blown essay with thousands of words
- and depending on the use case you’ll need to handle these extremes differently.

An alternative way to add new columns to a dataset is with the `Dataset.add_column()` function.

This allows you to provide the column as a Python **list or NumPy array** and can be handy in situations where `Dataset.map()` is not well suited for your analysis.

In [ ]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}
drug_dataset = drug_dataset.map(compute_review_length)
pprint(drug_dataset["train"][0])

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1,2, sharey=True, figsize = (10,5))
axs[0].hist(drug_dataset['train']['review_length'], bins=50, log=True);
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)
axs[1].hist(drug_dataset['train']['review_length'], bins=50, log=True);

4. Hnadle HTML characters in reviews.
- The last thing we need to deal with is the presence of HTML character codes in our reviews.
- We can use Python’s `html` module to unescape these characters.

In [ ]:
text = "I&#039;m a transformer called BERT"
html.unescape(text)

In [ ]:
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

## Batched `Dataset` ops
- If set to `True`, causes it to send a batch of examples to the map function at once
- The `batch_size` is configurable but defaults to 1,000.
- The received batch has the fields of the dataset, but each value is now a list of values, and not just a single value.
- The return value of `Dataset.map()` should be the same:
    - a dictionary with the fields we want to update or add to our dataset
    - and a list of values

### Pipeline time without batch

In [ ]:
# !rm -rf ~/.cache/huggingface/datasets/

In [ ]:
# data_files = {"train": "data/drugsComTrain_raw.tsv", "test": "data/drugsComTest_raw.tsv"}

# def lowercase_condition(example):
#     return {"condition": example["condition"].lower()}

# def compute_review_length(example):
#     return {"review_length": len(example["review"].split())}

# drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
# drug_dataset = drug_dataset.rename_column(
#     original_column_name="Unnamed: 0", new_column_name="patient_id"
# )

In [ ]:
# %%time
# drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)
# drug_dataset = drug_dataset.map(lowercase_condition)
# drug_dataset = drug_dataset.map(compute_review_length)
# drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
# drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

### Pipeline time with batch

In [ ]:
!rm -rf ~/.cache/huggingface/datasets/

In [ ]:
def lowercase_condition(batch):
    return {
        "condition": [c.lower() for c in batch["condition"]]
    }

def compute_review_length(batch):
    return {
        "review_length": [len(r.split()) for r in batch["review"]]
    }

drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
drug_dataset = drug_dataset.rename_column("Unnamed: 0", "patient_id")

In [ ]:
%%time
drug_dataset = drug_dataset.filter(
    lambda batch: [c is not None for c in batch["condition"]],
    batched=True, batch_size = 1000,
)
drug_dataset = drug_dataset.map(lowercase_condition, batched=True, batch_size = 1000)
drug_dataset = drug_dataset.map(compute_review_length, batched=True, batch_size = 1000)
drug_dataset = drug_dataset.filter(
    lambda batch: [rl > 30 for rl in batch["review_length"]],
    batched=True, batch_size = 1000,
)
drug_dataset = drug_dataset.map(
    lambda batch: {"review": [html.unescape(r) for r in batch["review"]]},
    batched=True, batch_size = 1000,
)

### Tokenization w/wo batch

Using `Dataset.map()` with `batched=True` will be essential to unlock the speed of the **“fast” tokenizers**

In [ ]:
from transformers import AutoTokenizer
fast_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["review"], truncation=True)
    # the tokenizer itself handle the list or iterable input values.

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=False, fn_kwargs={"tokenizer": fast_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": fast_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, num_proc=8, fn_kwargs={"tokenizer": fast_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=False, fn_kwargs={"tokenizer": slow_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": slow_tokenizer})

In [ ]:
# %time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True, num_proc=8, fn_kwargs={"tokenizer": slow_tokenizer})

- In general, it's not recommended to use Python multiprocessing for fast tokenizers with `batched=True`.
- Using `num_proc` to speed up your processing is usually a great idea, as long as the function you are using is not already doing some kind of multiprocessing of its own.

## Creating multiple recrods from a single record
- With `Dataset.map()` and `batched=True` you can change the number of elements in your dataset.
- This is super useful in many situations (like here and for question answering) where you want to create several training features from one example

- In the following example, we will tokenize our examples and truncate them to a maximum length of 128
- But we will ask the tokenizer to return all the chunks of the texts instead of just the first one.
- This can be done with `return_overflowing_tokens=True`:

In [ ]:
def tokenize_and_split(examples):
    return fast_tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )

result = tokenize_and_split(drug_dataset["train"][0])
[len(inp) for inp in result["input_ids"]]

In [ ]:
# tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
# # ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

- The problem is that we’re trying to mix two different datasets of different sizes:
    - The `drug_dataset` columns will have a certain number of examples (the 1,000 in our error)
    - But the `tokenized_dataset` we are building will have more (the 1,463 in the error message)
        - it is more than 1,000 because we are tokenizing long reviews into more than one example by using `return_overflowing_tokens=True`.

The solution:
- Remove the columns from the old dataset
    - with the `remove_columns` argument
- Make them the same size as they are in the new dataset.
    - Using `overflow_to_sample_mapping`

### First Option

In [ ]:
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset["train"].column_names
)

In [ ]:
print(tokenized_dataset["train"].features)
print(len(tokenized_dataset["train"]), len(drug_dataset["train"]))

### Second Option

In [ ]:
def tokenize_and_split(examples):
    result = fast_tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result

tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
tokenized_dataset

## From Datasets to DataFrames and back

- To enable the conversion between various third-party libraries, Datasets provides a `Dataset.set_format()` function.
- This function **only** changes the **output format** of the dataset
- So, you can easily switch to another format **without affecting** the underlying data format, which is **Apache Arrow**.
- The formatting is done in place.

In [ ]:
drug_dataset.set_format("pandas")
drug_dataset["train"][:3]

Creating a `pandas.DataFrame` for the whole training set by selecting all the elements of `drug_dataset["train"]`
- Under the hood, `Dataset.set_format()` changes the return format for the dataset’s `__getitem__()` dunder method.
- This means that when we want to create a new object like `train_df` from a Dataset in the "pandas" format
    - we need to slice the whole dataset to obtain a `pandas.DataFrame`.
- You can verify for yourself that the type of `drug_dataset["train"]` is `Dataset`, irrespective of the output format.

In [ ]:
train_df = drug_dataset["train"][:]
train_df.head()

In [ ]:
frequencies = (
    train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "count": "frequency"})
)

avg_rating_per_drug = train_df\
    .groupby("drugName")\
    .agg(avg_rating = ('rating', "mean"))\
    .reset_index()

display(frequencies.head())
display(avg_rating_per_drug.head())

Once we’re done with our Pandas analysis, we can always create a new `Dataset` object by using the `Dataset.from_pandas()` function as follows:

In [ ]:
from datasets import Dataset
freq_dataset = Dataset.from_pandas(frequencies)
drug_rating_dataset = Dataset.from_pandas(avg_rating_per_drug)
print(freq_dataset)
print(drug_rating_dataset)

reset the output format of `drug_dataset` from `"pandas"` to `"arrow"`

In [ ]:
drug_dataset.reset_format()

## Creating a validation set

Datasets provides a `Dataset.train_test_split()` function that is based on the famous functionality from `scikit-learn`.

In [ ]:
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
# Add the "test" set to our `DatasetDict`
drug_dataset_clean["test"] = drug_dataset["test"]
drug_dataset_clean

# Saving/Loading a Dataset to/from disk

- Datasets will cache every downloaded dataset and the operations performed on it
- Still, there are times when you’ll want to save a dataset to disk (e.g., in case the cache gets deleted).

`Dataset.save_to_disk()` --> saved as `Arrow`

`Dataset.to_csv()` --> saved as `CSV`

`Dataset.to_json()` --> saved as `JSON`


### Arrow format

In [ ]:
save_dir = "data/drug-reviews"
drug_dataset_clean.save_to_disk(save_dir)

In [ ]:
!tree {save_dir}

Once the dataset is saved, we can load it by using the `load_from_disk()` function as follows

In [ ]:
from datasets import load_from_disk

drug_dataset_reloaded = load_from_disk(save_dir)
drug_dataset_reloaded

### Json format

In [ ]:
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"data/drug-reviews-{split}.jsonl")

In [ ]:
!head -n 1 data/drug-reviews-train.jsonl

In [ ]:
data_files = {
    "train": "data/drug-reviews-train.jsonl",
    "validation": "data/drug-reviews-validation.jsonl",
    "test": "data/drug-reviews-test.jsonl",
}
drug_dataset_reloaded = load_dataset("json", data_files=data_files)
drug_dataset_reloaded

# TODOs
- Use the techniques from [Chapter 3](https://huggingface.co/course/chapter3) to train a classifier that can predict the **patient condition** based on the **drug review**.
    - Use `Trainer` API
    - The challenge, not addressed yet, is to prepare labels for the trainer. (keep track of label values and names)
- Use the `summarization` pipeline from [Chapter 1](https://huggingface.co/course/chapter1) to generate summaries of the **reviews**.
    - This should as efficient/batched as possible
    - Bonus: make it even parallel